# NB158 — The R0 Analytic Mass Formula

**Context**: x(R0) = 4/7 is exact (37 ppm) for the quark 1->2 step. The R0 level
has a closed-form analytic solution (NB138). This is the ONLY level where we can
work symbolically rather than numerically.

**Goal**: Starting from the R0 analytic formula, derive:
1. WHY x(R0) = 4/7 = phi(p3)/p4
2. What determines the 2->3 R0 exponents
3. Whether the scaling factors emerge from the analytic structure

**Method**: The R0 analytic formula gives RMS(ci) as a function of kappa, D, C1,
and the crossing position ci. The crossing positions come from CRT. We compute
the CP ratio and its logarithm symbolically to find what makes x = 4/7.

In [2]:
# S0 — The R0 analytic formula: symbolic decomposition
#
# R0(ci; j1) = (2*pi*j1 + D) * alpha(ci) - D
# where alpha(ci) = e^{-kappa*ci}, D = eps*omega/(omega^2 + kappa^2)
# kappa = eps = 1/sqrt(P4), omega = 2*pi
#
# At level R0, there are p1=2 sheets (j1=0 and j1=1).
# RMS(ci)^2 = 0.5 * [R0(ci,0)^2 + R0(ci,1)^2]
#           = 0.5 * [D^2*(a-1)^2 + (C1*a-D)^2]
#           where a = alpha(ci), C1 = 2*pi + D

import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)  # 210
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
epsilon = kappa
D = epsilon * omega / (omega**2 + kappa**2)
C1 = 2 * np.pi + D

P = [1]
for p in primes:
    P.append(P[-1] * p)
P3 = P[3]  # = 30

# The three quark Z2=0 generation crossings:
ci_g1, ci_g2, ci_g3 = 71, 11, 191
# Spacings: g2->g1 = 60 = 2*P3, g2->g3 = 180 = 6*P3, g1->g3 = 120 = 4*P3
# Equivalently (short way): g3->g2 = 30 = P3, g1->g2 = 60 = 2*P3, g3->g1 = 90 = 3*P3

print('=== R0 analytic parameters ===')
print(f'kappa = 1/sqrt({P4}) = {kappa}')
print(f'D = kappa*omega/(omega^2+kappa^2) = {D}')
print(f'C1 = 2*pi + D = {C1}')
print(f'D/kappa = omega/(omega^2+kappa^2) = {D/kappa}')
print(f'C1/D = 2*pi/D + 1 = {C1/D:.4f}')
print(f'  = omega^2/kappa^2 + 2 = {omega**2/kappa**2 + 2:.4f}')
print(f'  = 4*pi^2*P4 + 2 = {4*np.pi**2*P4 + 2:.4f}')

# KEY: D is very small (D ≈ kappa*omega/omega^2 = kappa/omega = 1/(omega*sqrt(P4)))
# So D ≈ 1/(2*pi*sqrt(210)) ≈ 0.011
# And C1 ≈ 2*pi (to leading order)
# C1/D ≈ 4*pi^2*P4 = very large

print(f'\nD ≈ 1/(2*pi*sqrt(P4)) = {1/(2*np.pi*np.sqrt(P4)):.6f} (exact: {D:.6f})')
print(f'C1 ≈ 2*pi = {2*np.pi:.6f} (exact: {C1:.6f})')

# ══════════════════════════════════════════════════════════════
# The RMS at each crossing
# ══════════════════════════════════════════════════════════════

def alpha(ci):
    return np.exp(-kappa * ci)

def rms_sq(ci):
    a = alpha(ci)
    r0 = D * (a - 1)
    r1 = C1 * a - D
    return 0.5 * (r0**2 + r1**2)

def rms(ci):
    return np.sqrt(rms_sq(ci))

# CP_R0^2 = RMS(g2)^2 / RMS(g3)^2
cp_sq = rms_sq(ci_g2) / rms_sq(ci_g3)
cp = np.sqrt(cp_sq)

print(f'\n=== RMS at generation crossings ===')
for label, ci in [('gen2', ci_g2), ('gen1', ci_g1), ('gen3', ci_g3)]:
    a = alpha(ci)
    r0 = D*(a-1)
    r1 = C1*a - D
    print(f'  {label} (ci={ci}): alpha={a:.10f} R0_j0={r0:.10f} R0_j1={r1:.10f} RMS={rms(ci):.10f}')

print(f'\nCP_R0 = {cp:.10f}')
print(f'ln(CP_R0) = {np.log(cp):.10f}')

# ══════════════════════════════════════════════════════════════
# SIMPLIFICATION: At ci=191 (gen3), alpha ≈ 0. So:
#   R0(191, j1) ≈ -D for both j1
#   RMS(191) ≈ D
#
# At ci=11 (gen2), alpha = 0.468. The j1=1 branch dominates:
#   R0(11, 1) = C1*alpha - D ≈ 2*pi*alpha (since D << C1*alpha)
#   R0(11, 0) = D*(alpha-1) ≈ -D (small)
#   RMS(11) ≈ |R0(11,1)| / sqrt(2) ≈ 2*pi*alpha / sqrt(2)
#
# So CP_R0 ≈ RMS(11)/RMS(191) ≈ (2*pi*alpha(11)/sqrt(2)) / D
#          = 2*pi*alpha/(sqrt(2)*D)
#          = 2*pi*e^{-11*kappa} / (sqrt(2) * kappa/(omega^2))
#          ... but this is approximate. Let me be more careful.
# ══════════════════════════════════════════════════════════════

a11 = alpha(ci_g2)  # alpha at ci=11
a71 = alpha(ci_g1)  # alpha at ci=71
a191 = alpha(ci_g3)  # alpha at ci=191

# Exact RMS^2 at ci=11:
# = 0.5 * [D^2*(a11-1)^2 + (C1*a11 - D)^2]
# = 0.5 * [D^2*a11^2 - 2*D^2*a11 + D^2 + C1^2*a11^2 - 2*C1*D*a11 + D^2]
# = 0.5 * [(D^2 + C1^2)*a11^2 - 2*D*(D + C1)*a11 + 2*D^2]
# = 0.5 * [(D^2 + C1^2)*a11^2 - 2*D*C1*a11 - 2*D^2*a11 + 2*D^2]
# Let me define: A = D^2 + C1^2,  B = 2*D*(D + C1),  C_const = 2*D^2
# Then RMS^2(ci) = 0.5 * [A*alpha^2 - B*alpha + C_const]

A_coeff = D**2 + C1**2
B_coeff = 2 * D * (D + C1)
C_coeff = 2 * D**2

print(f'\n=== Quadratic form in alpha ===')
print(f'RMS^2(ci) = 0.5 * [A*alpha^2 - B*alpha + C]')
print(f'A = D^2 + C1^2 = {A_coeff:.10f}')
print(f'B = 2*D*(D + C1) = {B_coeff:.10f}')
print(f'C = 2*D^2 = {C_coeff:.10f}')
print(f'A/C = (D^2+C1^2)/(2D^2) = {A_coeff/C_coeff:.4f}')
print(f'  ≈ C1^2/(2D^2) = {C1**2/(2*D**2):.4f}')
print(f'  ≈ 2*pi^2*P4^2 = {2*np.pi**2*P4**2:.4f} (since C1≈2pi, D≈1/(2pi*sqrt(P4)))')

# Verify: 
for ci in [11, 71, 191]:
    a = alpha(ci)
    rms2_formula = 0.5 * (A_coeff * a**2 - B_coeff * a + C_coeff)
    rms2_direct = rms_sq(ci)
    print(f'  ci={ci}: formula={rms2_formula:.10f} direct={rms2_direct:.10f} match={abs(rms2_formula-rms2_direct)<1e-15}')

# ══════════════════════════════════════════════════════════════
# CP_R0^2 = RMS^2(g2) / RMS^2(g3)
#         = [A*a11^2 - B*a11 + C] / [A*a191^2 - B*a191 + C]
#
# Since a191 ≈ 0:
#   denominator ≈ C = 2*D^2
#
# Since D << C1*a11:
#   numerator ≈ A*a11^2 = (D^2 + C1^2)*a11^2 ≈ C1^2*a11^2
#
# So CP_R0^2 ≈ C1^2*a11^2 / (2*D^2) = (C1*a11)^2 / (2*D^2)
# CP_R0 ≈ C1*a11 / (D*sqrt(2))
#       ≈ 2*pi * e^{-11*kappa} / (D*sqrt(2))
# ══════════════════════════════════════════════════════════════

print(f'\n=== CP_R0 simplified ===')
cp_approx_1 = C1 * a11 / (D * np.sqrt(2))
print(f'CP_R0 ≈ C1*alpha(11)/(D*sqrt(2)) = {cp_approx_1:.6f} (exact: {cp:.6f}, dev: {abs(cp_approx_1/cp-1)*100:.3f}%)')

# ln(CP_R0) ≈ ln(C1/D) + ln(alpha(11)) - ln(sqrt(2))
#           = ln(C1/D) - 11*kappa - 0.5*ln(2)
ln_cp_approx = np.log(C1/D) - 11*kappa - 0.5*np.log(2)
ln_cp_exact = np.log(cp)
print(f'ln(CP_R0) ≈ ln(C1/D) - 11*kappa - 0.5*ln(2) = {ln_cp_approx:.6f} (exact: {ln_cp_exact:.6f})')

# Now: x(R0) = ln(mass_ratio) / ln(CP_R0)
# For m_s/m_d = 20: x(R0) = ln(20) / [ln(C1/D) - 11*kappa - 0.5*ln(2)]
#
# C1/D ≈ omega^2/kappa^2 = 4*pi^2*P4 (leading order)
# ln(C1/D) ≈ ln(4*pi^2*P4) = 2*ln(2*pi) + ln(P4)

print(f'\n=== Decomposing ln(CP_R0) ===')
print(f'ln(C1/D) = {np.log(C1/D):.10f}')
print(f'  = ln(omega^2/kappa^2 + 2) = ln(4*pi^2*P4 + 2) = {np.log(4*np.pi**2*P4 + 2):.10f}')
print(f'11*kappa = 11/sqrt(P4) = {11*kappa:.10f}')
print(f'0.5*ln(2) = {0.5*np.log(2):.10f}')
print(f'Sum: {np.log(C1/D) - 11*kappa - 0.5*np.log(2):.10f}')
print(f'Exact ln(CP_R0) = {ln_cp_exact:.10f}')

# ══════════════════════════════════════════════════════════════
# THE KEY: x(R0) = ln(20) / ln(CP_R0(11, 191))
# But CP_R0 depends on the crossing positions ci=11 and ci=191.
# These positions come from CRT: they're determined by the primes.
#
# ci=11 is the gen2 Z2=0 quark crossing. In CRT:
#   11 mod 3 = 2 (a3=1 in the discrete log sense)
#   11 mod 5 = 1 (a5=0)
#   11 mod 7 = 4 (a7=4)
#
# ci=191 is the gen3 Z2=0 quark crossing.
#   191 mod 3 = 2 (a3=1)
#   191 mod 5 = 1 (a5=0)  
#   191 mod 7 = 2 (a7=2)
#
# The DIFFERENCE: 191 - 11 = 180 = 6 × 30 = 6 × P3
# In Z7: a7 changes from 4 to 2, a change of -2 mod 7 = +5 mod 7.
# In ci-space: this maps to 6*P3 = 180.
#
# The CP_R0 depends primarily on exp(-kappa*(ci_g3 - ci_g2)) = exp(-180*kappa)
# = exp(-180/sqrt(210))
# ══════════════════════════════════════════════════════════════

delta_ci = ci_g3 - ci_g2  # 180
print(f'\n=== Crossing difference structure ===')
print(f'ci_g2 = {ci_g2}, ci_g3 = {ci_g3}')
print(f'Delta = {delta_ci} = {delta_ci // P3} × P3 = {delta_ci // P3} × {P3}')
print(f'exp(-kappa*delta) = exp(-{delta_ci}/sqrt({P4})) = exp(-{delta_ci*kappa:.6f}) = {np.exp(-kappa*delta_ci):.6e}')

# In the limit alpha(g3) → 0, CP_R0^2 ≈ A*alpha(g2)^2 / C
# = (D^2 + C1^2) * exp(-2*kappa*ci_g2) / (2*D^2)
# ≈ C1^2/(2*D^2) * exp(-2*kappa*ci_g2)
#
# So ln(CP_R0) ≈ 0.5*ln(C1^2/(2D^2)) - kappa*ci_g2
#              = ln(C1/(D*sqrt(2))) - kappa*ci_g2
#
# And x(R0) = ln(20) / [ln(C1/(D*sqrt(2))) - kappa*ci_g2]
#
# With C1 ≈ 2*pi, D ≈ kappa/(2*pi) (leading order):
# C1/(D*sqrt(2)) ≈ 2*pi / (kappa/(2*pi) * sqrt(2)) = 4*pi^2 / (kappa*sqrt(2))
#                = 4*pi^2*sqrt(P4)/sqrt(2) = 4*pi^2*sqrt(P4/2)

print(f'\n=== Leading order: what determines x(R0)? ===')
print(f'ln(CP_R0) ≈ ln(C1/(D*sqrt(2))) - kappa*ci_g2')
print(f'          = ln(4*pi^2*sqrt(P4/2)) - ci_g2/sqrt(P4)')

# Let's define: L = ln(4*pi^2*sqrt(P4/2)) = ln(4*pi^2) + 0.5*ln(P4/2)
L = np.log(4*np.pi**2) + 0.5*np.log(P4/2)
print(f'L = ln(4*pi^2) + 0.5*ln(P4/2) = {np.log(4*np.pi**2):.6f} + {0.5*np.log(P4/2):.6f} = {L:.6f}')
print(f'kappa*ci_g2 = {kappa*ci_g2:.6f}')
print(f'L - kappa*ci_g2 = {L - kappa*ci_g2:.6f}')
print(f'Exact ln(CP_R0) = {ln_cp_exact:.6f}')
print(f'Approx deviation: {abs(L - kappa*ci_g2 - ln_cp_exact)/ln_cp_exact*100:.3f}%')

# x(R0) ≈ ln(20) / (L - ci_g2/sqrt(P4))
x_approx = np.log(20) / (L - ci_g2/np.sqrt(P4))
print(f'\nx(R0) ≈ ln(20) / (L - ci_g2/sqrt(P4)) = {x_approx:.8f}')
print(f'4/7 = {4/7:.8f}')
print(f'Deviation: {abs(x_approx - 4/7)/(4/7)*100:.4f}%')

# ══════════════════════════════════════════════════════════════
# CRITICAL: What if we compute x(R0) for DIFFERENT crossing pairs
# using the SAME analytic formula?
# The mass ratio for each pair is:
#   mass_ratio = [RMS(ci_a)/RMS(ci_b)]^{x(R0)}
# But x(R0) was DEFINED as ln(m_s/m_d)/ln(CP_R0) for the CP pair.
#
# What if we instead compute the ACTUAL mass ratio that the
# R0 analytic formula PREDICTS for each generation pair,
# using x(R0) = 4/7 as the universal exponent?
# ══════════════════════════════════════════════════════════════

print(f'\n{"="*70}')
print(f'ACTUAL R0 PREDICTIONS with x(R0) = 4/7')
print(f'{"="*70}')

x_R0 = 4.0 / 7.0

# The CP pair gives m_s/m_d:
cp_23 = rms(ci_g2) / rms(ci_g3)
m_sd = cp_23 ** x_R0
print(f'\nCP pair (gen2/gen3, ci=11/191):')
print(f'  CP = {cp_23:.6f}, CP^(4/7) = {m_sd:.4f} [PDG m_s/m_d = 20.0]')

# But what about gen1 (ci=71)? Its R0 profile is different.
# The gen1 crossing sits BETWEEN gen2 and gen3 in the spatial profile.
# If mass ∝ RMS^{-4/7}, what mass does gen1 predict?

# The idea: EACH generation crossing has a spatial amplitude.
# The mass of the fermion at that crossing is determined by that amplitude.
# Specifically: mass ∝ [RMS(ci)]^{-4/7}
# (inverse because smaller RMS = heavier fermion, as gen3 shows)

# Wait — gen2 has the LARGEST RMS but maps to the MIDDLE mass (s quark).
# This doesn't work as a direct proportionality.

# The CP pair formula works because it's a RATIO at specific crossings.
# The mass is not proportional to 1/RMS at a single crossing —
# it's the RATIO of RMS values at the CP pair raised to 4/7.

# But can we compute other mass ratios from other pairs?
# m_b/m_d should involve gen3 and gen1 somehow.

# Actually: in the current pipeline,
# m_b/m_s uses the CP pair (gen2/gen3) with a DIFFERENT exponent (r_bs*x_q).
# What if m_b/m_d uses a different PAIR with the SAME exponent (4/7)?

# There are 3 gen crossings and 3 possible pairs:
# gen2/gen3 → m_s/m_d = 20.0
# gen?/gen? → m_b/m_s = 44.75
# gen?/gen? → m_b/m_d = 895

# The product: m_b/m_d = m_b/m_s × m_s/m_d = 44.75 × 20 = 895.
# And (gen2/gen3)^{4/7} × (something)^{4/7} = 895?
# 20 × something = 895 → something = 44.75 = m_b/m_s.
# We need a pair whose R0 ratio raised to 4/7 gives 44.75.

# Check ALL possible R0 ratios^{4/7}:
print(f'\nAll R0 ratio^(4/7) between gen crossings:')
gen_data = {'gen1': ci_g1, 'gen2': ci_g2, 'gen3': ci_g3}
for na, cia in gen_data.items():
    for nb, cib in gen_data.items():
        if cia == cib:
            continue
        ratio = rms(cia) / rms(cib)
        val = ratio ** x_R0 if ratio > 0 else 0
        print(f'  ({na}/{nb})^(4/7) = ({rms(cia)/rms(cib):.4f})^(4/7) = {val:.4f}')

# ══════════════════════════════════════════════════════════════
# Include the Z2=1 crossings too!
# gen1 Z2=1: ci=41, gen2 Z2=1: ci=101, gen3 Z2=1: ci=131
# ══════════════════════════════════════════════════════════════

ci_z1 = {'gen1_z1': 41, 'gen2_z1': 101, 'gen3_z1': 131}
all_ci = {**gen_data, **ci_z1}

print(f'\nAll R0 ratios^(4/7) including Z2=1 crossings:')
print(f'{"pair":>20}  {"R0_ratio":>10}  {"^(4/7)":>10}  {"matches?":>15}')
targets = {'m_s/m_d': 20.0, 'm_b/m_s': 44.75, 'm_t/m_c': 135.8, 'm_b/m_d': 895, 'm_c/m_u': 588}

for na, cia in sorted(all_ci.items()):
    for nb, cib in sorted(all_ci.items()):
        if cia >= cib:
            continue
        ratio = rms(cia) / rms(cib)
        val = ratio ** x_R0
        # Check against targets
        match = ''
        for tname, tval in targets.items():
            if abs(val - tval) / tval < 0.05:  # within 5%
                match = f'{tname} ({abs(val-tval)/tval*100:.1f}%)'
        if match or val > 10:
            print(f'  {na}/{nb:>10}  {ratio:10.4f}  {val:10.2f}  {match}')

# ══════════════════════════════════════════════════════════════
# PRODUCTS of ratios
# ══════════════════════════════════════════════════════════════

print(f'\n=== Products of R0 ratios^(4/7) ===')
print(f'Target: m_b/m_s = 44.75')

# m_b/m_s might be a PRODUCT of R0 ratios from different crossing pairs
# Check: (gen2/gen3)^{4/7} × (genX/genY)^{4/7} = 44.75?
# → (genX/genY)^{4/7} = 44.75/20 = 2.2375
# → genX/genY = 2.2375^{7/4} = 2.2375^1.75 = ?
target_ratio = (44.75/20.0) ** (7.0/4.0)  # what R0 ratio would we need
print(f'  Need R0 ratio = (44.75/20)^(7/4) = {target_ratio:.4f}')

# Check all pairs for this ratio
for na, cia in sorted(all_ci.items()):
    for nb, cib in sorted(all_ci.items()):
        if cia == cib:
            continue
        ratio = rms(cia) / rms(cib)
        if abs(ratio - target_ratio) / target_ratio < 0.1:
            print(f'  {na}/{nb}: R0 ratio = {ratio:.4f} (need {target_ratio:.4f}, dev {abs(ratio-target_ratio)/target_ratio*100:.1f}%)')

# What if m_b/m_s involves Z2=0 AND Z2=1 crossings?
# m_b/m_s = [(gen2_z0/gen3_z0) × (gen1_z1/gen3_z1)]^{4/7} ?
# = [CP_pair × (gen1_z1/gen3_z1)]^{4/7}
r_z1_13 = rms(41) / rms(131)
product_test = (cp_23 * r_z1_13) ** x_R0
print(f'\n  (CP × gen1_z1/gen3_z1)^(4/7) = ({cp_23:.4f} × {r_z1_13:.4f})^(4/7) = {product_test:.4f}  [target: 44.75]')

# What about (gen2_z0 × gen1_z1) / (gen3_z0 × gen3_z1)?
cross_ratio = (rms(11) * rms(41)) / (rms(191) * rms(131))
print(f'  (gen2_z0 × gen1_z1)/(gen3_z0 × gen3_z1) = {cross_ratio:.4f}  ^(4/7) = {cross_ratio**x_R0:.4f}')

# Or: product of Z2=0 and Z2=1 CP pairs?
cp_z0 = rms(11) / rms(191)  # gen2/gen3 Z2=0
cp_z1 = rms(101) / rms(131)  # gen2/gen3 Z2=1 (if applicable)
print(f'\n  Z2=0 CP = {cp_z0:.4f}  Z2=1 "CP" = {cp_z1:.4f}')
print(f'  (Z2=0 CP)^(4/7) = {cp_z0**x_R0:.4f}')
print(f'  (Z2=1 "CP")^(4/7) = {cp_z1**x_R0:.4f}')
print(f'  Product = {cp_z0**x_R0 * cp_z1**x_R0:.4f}  [target m_b/m_s = 44.75]')
print(f'  (Z2=0 × Z2=1)^(4/7) = {(cp_z0 * cp_z1)**x_R0:.4f}')

=== R0 analytic parameters ===
kappa = 1/sqrt(210) = 0.06900655593423542
D = kappa*omega/(omega^2+kappa^2) = 0.010981409900003394
C1 = 2*pi + D = 6.294166717079589
D/kappa = omega/(omega^2+kappa^2) = 0.15913574806528366
C1/D = 2*pi/D + 1 = 573.1656
  = omega^2/kappa^2 + 2 = 8292.4677
  = 4*pi^2*P4 + 2 = 8292.4677

D ≈ 1/(2*pi*sqrt(P4)) = 0.010983 (exact: 0.010981)
C1 ≈ 2*pi = 6.283185 (exact: 6.294167)

=== RMS at generation crossings ===
  gen2 (ci=11): alpha=0.4681005689 R0_j0=-0.0058410057 R0_j1=2.9353216113 RMS=2.0755899257
  gen1 (ci=71): alpha=0.0074505645 R0_j0=-0.0108995922 R0_j1=0.0359136855 RMS=0.0265385937
  gen3 (ci=191): alpha=0.0000018875 R0_j0=-0.0109813892 R0_j1=-0.0109695296 RMS=0.0109754610

CP_R0 = 189.1118676498
ln(CP_R0) = 5.2423387323

=== Quadratic form in alpha ===
RMS^2(ci) = 0.5 * [A*alpha^2 - B*alpha + C]
A = D^2 + C1^2 = 39.6166552538
B = 2*D*(D + C1) = 0.1384788321
C = 2*D^2 = 0.0002411827
A/C = (D^2+C1^2)/(2D^2) = 164259.9194
  ≈ C1^2/(2D^2) = 164259.4194
